# 02 - CSV para Delta Lake (MinIO)

Lê os arquivos CSV do bucket `landing-zone` no MinIO e grava cada um como tabela Delta Lake no bucket `bronze`.

**Pré-requisitos:** Notebook `01` executado (CSVs no MinIO).

## 1. Imports e Configuração

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://minio:9000
Landing: landing-zone | Bronze: bronze


## 2. Criar SparkSession com Delta Lake e MinIO

In [2]:
spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-51208664-ab54-4dd7-bc6a-5cb6ea8531f9;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (505ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (8

SparkSession criada com sucesso!


## 3. Criar Bucket Bronze no MinIO

In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado!
Buckets: ['bronze', 'landing-zone']


## 4. Listar CSVs Disponíveis no Landing Zone

In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

7 arquivos CSV encontrados no bucket [landing-zone]:
  - anexo.csv
  - cidade.csv
  - estado.csv
  - ouvidoria.csv
  - servico_afetado.csv
  - tipo_ouvidoria.csv
  - usuario.csv


## 5. Ler CSVs e Gravar como Delta Lake

In [5]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)
    
    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)
    
    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

Convertendo 7 CSVs para Delta Lake...



26/05/01 23:08:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


  anexo: 2 registros | 4 colunas -> s3a://bronze/anexo
  cidade: 500 registros | 3 colunas -> s3a://bronze/cidade
  estado: 27 registros | 3 colunas -> s3a://bronze/estado
  ouvidoria: 3 registros | 7 colunas -> s3a://bronze/ouvidoria
  servico_afetado: 8 registros | 2 colunas -> s3a://bronze/servico_afetado
  tipo_ouvidoria: 5 registros | 3 colunas -> s3a://bronze/tipo_ouvidoria
  usuario: 3 registros | 11 colunas -> s3a://bronze/usuario

Conversao concluida! 7 tabelas Delta criadas no bucket [bronze].


## 6. Validação - Ler Tabelas Delta

In [6]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    
    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake...



26/05/01 23:09:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/01 23:09:10 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


  anexo: Delta=True | 2 registros | Colunas: ['id_anexo', 'nome_anexo', 'arquivo_anexo', 'cod_ouvidoria']
  cidade: Delta=True | 500 registros | Colunas: ['id_cidade', 'nome_cidade', 'cod_estado']
  estado: Delta=True | 27 registros | Colunas: ['id_estado', 'nome_estado', 'sigla_estado']
  ouvidoria: Delta=True | 3 registros | Colunas: ['id_ouvidoria', 'descricao_ouvidoria', 'cod_tipo', 'cod_servico', 'protocolo_ouvidoria', 'data_ouvidoria', 'cod_usuario']
  servico_afetado: Delta=True | 8 registros | Colunas: ['id_servico', 'nome_servico']
  tipo_ouvidoria: Delta=True | 5 registros | Colunas: ['id_tipo', 'nome_tipo', 'descricao_tipo']
  usuario: Delta=True | 3 registros | Colunas: ['id_usuario', 'nome_usuario', 'cpf_usuario', 'email_usuario', 'senha_usuario', 'telefone_usuario', 'whatsapp_usuario', 'data_nasc', 'cod_cidade', 'data_ultimo_acesso', 'hash_ativacao_usuario']


In [7]:
# Amostra: exibir primeiros registros de algumas tabelas
for tabela in ['cidade', 'servico_afetado', 'ouvidoria']:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- CIDADE ---
+---------+------------------+----------+
|id_cidade|       nome_cidade|cod_estado|
+---------+------------------+----------+
|        1|    Afonso Cláudio|         8|
|        2|Água Doce do Norte|         8|
|        3|      Águia Branca|         8|
|        4|            Alegre|         8|
|        5|    Alfredo Chaves|         8|
+---------+------------------+----------+
only showing top 5 rows


--- SERVICO_AFETADO ---
+----------+--------------------+
|id_servico|        nome_servico|
+----------+--------------------+
|         1|Educação, Ciência...|
|         2|               Saúde|
|         3|        Procuradoria|
|         4|             Fazenda|
|         5|         Agricultura|
+----------+--------------------+
only showing top 5 rows


--- OUVIDORIA ---
+------------+--------------------+--------+-----------+-------------------+-------------------+-----------+
|id_ouvidoria| descricao_ouvidoria|cod_tipo|cod_servico|protocolo_ouvidoria|     data_ouvidoria|c

In [8]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
